In [ ]:
# 8. Standalone Test Cases - # Test the mapping pipeline with the five classifier emotions.
test_emotions = ['sadness', 'anger', 'joy', 'anxiety', 'calm']

for emotion in test_emotions:
    result = recommend_songs(emotion_label=emotion, n_results=5)
    print(f"\n=== Emotion: {emotion} ===")
    print('Mood category:', result['mood_category'])
    print('Valence      :', result['valence'])
    print('Arousal      :', result['arousal'])
    for i, song in enumerate(result['songs'], start=1):
        print(f"{i}. {song['title']} - {song['artist']}")
        print('   Genre :', song['genre'])
        print('   Link  :', song['spotify_url'])

# 9. Example API-Ready Output - Build a clean array payload that can be returned later by Flask.
example_payload = recommend_songs(emotion_label='joy', n_results=5)['songs']
example_payload

In [ ]:
# ==============================
# 0. Imports
# Load the cleaned music dataset and prepare the mood mapping pipeline.
# ==============================
import ast
import random

import pandas as pd

In [2]:
# ==============================
# 1. Configuration
# Define the Kaggle dataset path and random seed.
# ==============================
DATA_PATH = '/kaggle/input/datasets/melisaolivia/nlp-music-music-cleaned/cleaned_spotify_lyrics.csv'
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print('Music dataset :', DATA_PATH)

Music dataset : /kaggle/input/datasets/melisaolivia/nlp-music-music-cleaned/cleaned_spotify_lyrics.csv


In [3]:
# ==============================
# 2. Load Dataset
# Read the cleaned Spotify lyrics dataset and inspect the available columns.
# ==============================
df = pd.read_csv(DATA_PATH)
print('Rows   :', len(df))
print('Columns:', list(df.columns))
print(df.head())

Rows   : 1547
Columns: ['song_id_clean', 'title', 'artists', 'genres_clean', 'mood_category', 'emotion_label', 'clean_text', 'final_text', 'spotify_url']
            song_id_clean            title             artists  \
0  0Gr8wdN1Rz6zQPcgcHOhK8  Out Here Really               Mozzy   
1  2PTa6TM7TDu8f6pFij0WI8      Perry Mason       Ozzy Osbourne   
2  4EbKYtGgDbX6fGNBS3kpzQ       Santa Baby      Colbie Caillat   
3  5sQeyJ1kOsqR3mUZNJZy5o      Stars Dance        Selena Gomez   
4  4Gc6bvNR0GBitjk6C83xxP          Electra  Marianne Faithfull   

                              genres_clean mood_category emotion_label  \
0    cali rap; pop rap; sacramento hip hop       intense         anger   
1  album rock; birmingham metal; hard rock       intense         anger   
2               dance pop; neo mellow; pop      euphoric           joy   
3            dance pop; pop; post-teen pop      euphoric           joy   
4         art rock; british invasion; folk   melancholic       sadness   

    

In [4]:
# ==============================
# 3. Clean and Standardize Columns
# Keep only the fields needed for recommendation and normalize text values.
# ==============================
required_columns = [
    'title',
    'artists',
    'genres_clean',
    'mood_category',
    'emotion_label',
    'spotify_url',
]

missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

work_df = df[required_columns].copy()

for col in ['title', 'artists', 'genres_clean', 'mood_category', 'emotion_label', 'spotify_url']:
    work_df[col] = work_df[col].fillna('').astype(str).str.strip()

work_df['mood_category'] = work_df['mood_category'].str.lower()
work_df['emotion_label'] = work_df['emotion_label'].str.lower()
work_df = work_df[(work_df['title'] != '') & (work_df['artists'] != '') & (work_df['spotify_url'] != '')]
work_df = work_df.drop_duplicates(subset=['title', 'artists', 'spotify_url']).reset_index(drop=True)

print('Usable rows:', len(work_df))
print(work_df.head())

Usable rows: 1547
             title             artists  \
0  Out Here Really               Mozzy   
1      Perry Mason       Ozzy Osbourne   
2       Santa Baby      Colbie Caillat   
3      Stars Dance        Selena Gomez   
4          Electra  Marianne Faithfull   

                              genres_clean mood_category emotion_label  \
0    cali rap; pop rap; sacramento hip hop       intense         anger   
1  album rock; birmingham metal; hard rock       intense         anger   
2               dance pop; neo mellow; pop      euphoric           joy   
3            dance pop; pop; post-teen pop      euphoric           joy   
4         art rock; british invasion; folk   melancholic       sadness   

                                         spotify_url  
0  https://open.spotify.com/track/0Gr8wdN1Rz6zQPc...  
1  https://open.spotify.com/track/2PTa6TM7TDu8f6p...  
2  https://open.spotify.com/track/4EbKYtGgDbX6fGN...  
3  https://open.spotify.com/track/5sQeyJ1kOsqR3mU...  
4  https:

In [5]:
# ==============================
# 4. Emotion to Valence-Arousal Mapping
# Map the classifier output to affective coordinates.
# ==============================
EMOTION_TO_VA = {
    'sadness': {'valence': -0.75, 'arousal': -0.40},
    'anger': {'valence': -0.80, 'arousal': 0.75},
    'joy': {'valence': 0.85, 'arousal': 0.70},
    'anxiety': {'valence': -0.65, 'arousal': 0.85},
    'calm': {'valence': 0.60, 'arousal': -0.55},
}

print(EMOTION_TO_VA)

{'sadness': {'valence': -0.75, 'arousal': -0.4}, 'anger': {'valence': -0.8, 'arousal': 0.75}, 'joy': {'valence': 0.85, 'arousal': 0.7}, 'anxiety': {'valence': -0.65, 'arousal': 0.85}, 'calm': {'valence': 0.6, 'arousal': -0.55}}


In [6]:
# ==============================
# 5. Valence-Arousal to Mood Mapping
# Convert the affective position into the project mood categories.
# ==============================
def valence_arousal_to_mood(valence, arousal):
    if valence < 0 and arousal < 0:
        return 'melancholic'
    if valence < 0 and arousal >= 0:
        if arousal >= 0.7:
            return 'tense'
        return 'intense'
    if valence >= 0 and arousal >= 0:
        return 'euphoric'
    return 'peaceful'


def emotion_to_mood(emotion_label):
    emotion_label = str(emotion_label).strip().lower()
    if emotion_label not in EMOTION_TO_VA:
        raise ValueError(f'Unsupported emotion label: {emotion_label}')

    va = EMOTION_TO_VA[emotion_label]
    mood = valence_arousal_to_mood(va['valence'], va['arousal'])
    return {
        'emotion': emotion_label,
        'valence': va['valence'],
        'arousal': va['arousal'],
        'mood_category': mood,
    }

for emotion in EMOTION_TO_VA:
    print(emotion_to_mood(emotion))

{'emotion': 'sadness', 'valence': -0.75, 'arousal': -0.4, 'mood_category': 'melancholic'}
{'emotion': 'anger', 'valence': -0.8, 'arousal': 0.75, 'mood_category': 'tense'}
{'emotion': 'joy', 'valence': 0.85, 'arousal': 0.7, 'mood_category': 'euphoric'}
{'emotion': 'anxiety', 'valence': -0.65, 'arousal': 0.85, 'mood_category': 'tense'}
{'emotion': 'calm', 'valence': 0.6, 'arousal': -0.55, 'mood_category': 'peaceful'}


In [7]:
# ==============================
# 6. Helper Filters
# Expand classifier emotions into related dataset emotions for better recall.
# ==============================
def allowed_song_emotions(emotion_label):
    emotion_label = str(emotion_label).strip().lower()
    mapping = {
        'sadness': ['sad', 'lonely', 'guilty', 'disappointed', 'devastated', 'heartbroken'],
        'anger': ['angry', 'furious', 'annoyed', 'disgusted'],
        'joy': ['joyful', 'excited', 'proud', 'grateful', 'hopeful', 'happy'],
        'anxiety': ['anxious', 'afraid', 'terrified', 'apprehensive', 'nervous'],
        'calm': ['content', 'peaceful', 'calm', 'confident', 'prepared', 'sentimental'],
    }
    return mapping.get(emotion_label, [emotion_label])


def parse_genres(value):
    value = str(value).strip()
    if not value:
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip().lower() for item in parsed if str(item).strip()]
    except Exception:
        pass

    value = value.replace('|', ',').replace(';', ',')
    return [part.strip().lower() for part in value.split(',') if part.strip()]

In [8]:
# ==============================
# 7. Recommendation Function
# Return 5 songs based on classifier emotion, mapped mood, and optional genre.
# ==============================
def recommend_songs(emotion_label, genre=None, n_results=5):
    mood_info = emotion_to_mood(emotion_label)
    target_mood = mood_info['mood_category']
    target_emotions = allowed_song_emotions(emotion_label)

    candidates = work_df[work_df['mood_category'] == target_mood].copy()

    if not candidates.empty:
        emotion_filtered = candidates[candidates['emotion_label'].isin(target_emotions)]
        if not emotion_filtered.empty:
            candidates = emotion_filtered

    if genre:
        genre = str(genre).strip().lower()
        genre_mask = candidates['genres_clean'].apply(lambda x: genre in parse_genres(x) or genre in str(x).lower())
        genre_filtered = candidates[genre_mask].copy()
        if not genre_filtered.empty:
            candidates = genre_filtered

    if candidates.empty:
        candidates = work_df[work_df['mood_category'] == target_mood].copy()

    if candidates.empty:
        raise ValueError(f'No songs found for mood category: {target_mood}')

    sample_size = min(n_results, len(candidates))
    sampled = candidates.sample(sample_size, random_state=random.randint(0, 10000)).copy()

    results = []
    for _, row in sampled.iterrows():
        results.append({
            'title': row['title'],
            'artist': row['artists'],
            'genre': row['genres_clean'],
            'mood_category': row['mood_category'],
            'spotify_url': row['spotify_url'],
        })

    return {
        'input_emotion': emotion_label,
        'valence': mood_info['valence'],
        'arousal': mood_info['arousal'],
        'mood_category': target_mood,
        'songs': results,
    }

In [ ]:
# ==============================
# 8. Standalone Test Cases
# Test the mapping pipeline with the five classifier emotions.
# ==============================
test_emotions = ['sadness', 'anger', 'joy', 'anxiety', 'calm']

for emotion in test_emotions:
    result = recommend_songs(emotion_label=emotion, n_results=5)
    print(f"\n=== Emotion: {emotion} ===")
    print('Mood category:', result['mood_category'])
    print('Valence      :', result['valence'])
    print('Arousal      :', result['arousal'])
    for i, song in enumerate(result['songs'], start=1):
        print(f"{i}. {song['title']} - {song['artist']}")
        print('   Genre :', song['genre'])
        print('   Link  :', song['spotify_url'])


=== Emotion: sadness ===
Mood category: melancholic
Valence      : -0.75
Arousal      : -0.4
1. However Long - Jason Isbell and the 400 Unit
   Genre : alternative country; deep new americana; folk
   Link  : https://open.spotify.com/track/0w1YTDoZnfJZMieR8zqPmT
2. Stand Back Up - Sugarland
   Genre : contemporary country; country; country dawn
   Link  : https://open.spotify.com/track/3PftBVjYwb3UiveoBhHHeO
3. No Choice in the Matter - Jason Isbell and the 400 Unit
   Genre : alternative country; deep new americana; folk
   Link  : https://open.spotify.com/track/5ZlJFhFdiAmsOMXifcIn2s
4. Link in Bio - Diet Cig
   Genre : bubblegrunge; indie pop; indie punk
   Link  : https://open.spotify.com/track/70qZCcbcsWAds3GzTqC4Sm
5. Latter Days - Big Red Machine, {label: spotify:artist:7K5Lm5dxoEwEpOS0Fc3l3s}
   Genre : eau claire indie; folktronica; indie folk
   Link  : https://open.spotify.com/track/7wOdJjz1WHJiBdzKSZWszu

=== Emotion: anger ===
Mood category: tense
Valence      : -0.8
Arou

In [10]:
# ==============================
# 9. Example API-Ready Output
# Build a clean array payload that can be returned later by Flask.
# ==============================
example_payload = recommend_songs(emotion_label='joy', n_results=5)['songs']
example_payload

[{'title': 'Umbrella',
  'artist': 'Rihanna, JAY-Z',
  'genre': 'barbadian pop; dance pop; pop',
  'mood_category': 'euphoric',
  'spotify_url': 'https://open.spotify.com/track/1YiD079ne2nl9TEOrw43Gh'},
 {'title': 'Love That Hurts',
  'artist': 'ZHU, {label: spotify:artist:4APHks5GcW1GrlfnSvL5O3}, {label: spotify:artist:7a5Srm7U661DotL6VWRmYk}',
  'genre': 'edm; electro house',
  'mood_category': 'euphoric',
  'spotify_url': 'https://open.spotify.com/track/5kBUIe8UcUUGLzP4hoc8XJ'},
 {'title': 'I Feel It Coming',
  'artist': 'The Weeknd, {label: spotify:artist:4tZwfgrHOc3mvqYlEYSvVi}',
  'genre': 'canadian contemporary r&b; canadian pop; pop',
  'mood_category': 'euphoric',
  'spotify_url': 'https://open.spotify.com/track/3dhjNA0jGA8vHBQ1VdD6vV'},
 {'title': 'National Anthem',
  'artist': 'Lana Del Rey',
  'genre': 'art pop; pop',
  'mood_category': 'euphoric',
  'spotify_url': 'https://open.spotify.com/track/0bhjD5O1sXKCdRbk9vz1a9'},
 {'title': 'like that',
  'artist': 'Jessica Baio',
